In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold


In [9]:
# Load the dataset
train = pd.read_csv("../artifacts/playground-series-s6e2/train.csv")
test = pd.read_csv("../artifacts/playground-series-s6e2/test.csv")
sample = pd.read_csv("../artifacts/playground-series-s6e2/sample_submission.csv")

train.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [10]:
X = train.drop(columns=["id", "Heart Disease"],axis=1)
y = train["Heart Disease"]
X_test = test.drop(columns=["id"],axis=1)

In [11]:
categorical_features = [
    "Chest pain type",
    "Thallium",
    "Slope of ST",
    "Exercise angina",
    "Sex",
    "FBS over 120",
    "EKG results",
    "Number of vessels fluro"
]

In [12]:
for col in categorical_features:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [15]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

In [16]:
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "n_estimators": 5000,
    "learning_rate": 0.01,
    "num_leaves": 128,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1
}

In [17]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    
    print(f"\n========== Fold {fold+1} ==========")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = lgb.LGBMClassifier(**params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        categorical_feature=categorical_features,
        callbacks=[lgb.early_stopping(300)]
    )
    
    # OOF predictions
    oof[valid_idx] = model.predict_proba(X_valid)[:, 1]
    
    # Test predictions
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits


========== Fold 1 ==========
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006655 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 426
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 13
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[1473]	valid_0's auc: 0.955447

========== Fold 2 ==========
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [

In [18]:
cv_score = roc_auc_score(y, oof)
print("\nFINAL CV AUC:", cv_score)


FINAL CV AUC: 0.9551560807160284


In [21]:
submission = pd.DataFrame({
    "id":test["id"],
    "Heart Disease": test_preds
})

submission.to_csv("submission.csv", index=False)
submission.head()

,id,Heart Disease
0,630000,0.953191
1,630001,0.009333
2,630002,0.985251
3,630003,0.006215
4,630004,0.172629
